# Scaling
**Goal:** Process raw market data into a comprehensive feature set for ML training.

1.  **Data Validation:** Flag features with distribution drifts or high NaN rates.
2.  **Scaling:** Apply Robust Z-scaling and Time-features relative to market sessions.
3.  
**Output:** Saves a fully featured, scaled dataset to `feat_all.parquet`.

In [1]:
# 1) WIPE & PURGE
%reset -f
import gc
import torch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# 2) IMPORTS
import os
import psutil
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from IPython.display import display

# Standardize display
pd.set_option('display.max_columns', None)

# 3) CUSTOM LIBRARIES
from libs import preps, plots, params, feats
importlib.reload(preps)
importlib.reload(plots)
importlib.reload(params)
importlib.reload(feats)

print(f"🚀 Memory Cleaned. Environment Ready for {params.ticker}")

🚀 Memory Cleaned. Environment Ready for AAPL


In [2]:
# Load raw data from the previous notebook's output

df_inds_unsc = pd.read_parquet(params.indunsc_parquet)

df_inds_unsc

,open,high,low,close,volume,trade_count,vwap,ask,bid,ret,log_ret,body,upper_shad,lower_shad,range_pct,sma_5,sma_pct_5,sma_9,sma_pct_9,sma_21,sma_pct_21,sma_50,sma_pct_50,sma_100,sma_pct_100,ema_3,ema_5,ema_9,ema_13,ema_21,ema_50,rsi_3,rsi_7,rsi_14,rsi_21,roc_3,roc_5,roc_10,roc_21,cci_7,cci_14,cci_20,macd_line_3_10_4,macd_signal_3_10_4,macd_diff_3_10_4,macd_line_6_13_5,macd_signal_6_13_5,macd_diff_6_13_5,macd_line_12_26_9,macd_signal_12_26_9,macd_diff_12_26_9,stoch_k_5_3_3,stoch_d_5_3_3,stoch_k_14_3_3,stoch_d_14_3_3,atr_3,atr_pct_3,plus_di_3,minus_di_3,adx_3,atr_7,atr_pct_7,plus_di_7,minus_di_7,adx_7,atr_14,atr_pct_14,plus_di_14,minus_di_14,adx_14,atr_21,atr_pct_21,plus_di_21,minus_di_21,adx_21,bb_lband_20_2p0,bb_hband_20_2p0,bb_w_20_2p0,bb_lband_20_3p0,bb_hband_20_3p0,bb_w_20_3p0,bb_lband_50_2p0,bb_hband_50_2p0,bb_w_50_2p0,kc_mid_20_20_1.5,kc_l_20_20_1.5,kc_h_20_20_1.5,kc_w_20_20_1.5,kc_mid_20_20_2.0,kc_l_20_20_2.0,kc_h_20_20_2.0,kc_w_20_20_2.0,obv,obv_roll_3,obv_roll_7,obv_roll_21,mfi_7,mfi_14,mfi_20,cmf_7,cmf_14,cmf_20,vol_spike_3,vol_spike_7,vol_spike_14,vol_spike_28,donch_h_10,donch_l_10,donch_w_10,donch_h_20,donch_l_20,donch_w_20,donch_h_55,donch_l_55,donch_w_55,roll_vwap_10,roll_vwap_20,roll_vwap_50,slope_close_5,slope_close_20,slope_close_50,ret_std_5,ret_std_21,ret_std_63,vwap_ohlc_close_session,vwap_dist_session,psar,psar_dist,dist_high_100,dist_low_100
2016-01-04 08:47:00,25.9500,25.9500,25.9500,25.9500,520.0,2.0,103.800000,25.955200,25.944800,0.000241,0.000241,0.0000,0.0000,0.0000,0.000000,25.93750,0.000482,25.933056,0.000653,25.926310,0.000914,25.938275,0.000452,26.147824,-0.007566,25.944217,25.940253,25.935579,25.932864,25.933197,25.987678,96.817308,90.125845,59.208009,43.243449,0.000723,0.000868,0.000868,0.001109,131.154684,224.437928,262.411348,0.009462,0.006510,0.002952,0.005934,0.003808,0.002126,-0.004585,-0.011539,0.006954,1.000000,0.888889,1.000000,0.971429,0.005205,0.000201,96.817213,3.182669,85.632759,0.003662,0.000141,87.291639,9.110761,46.159542,0.006309,0.000243,38.254340,24.700467,34.286782,0.009983,0.000385,21.487845,29.599048,51.826216,25.909948,25.943802,0.001306,25.901485,25.952265,0.001959,25.759513,26.117037,0.013784,25.932658,25.918347,25.946969,0.001104,25.932658,25.913577,25.951739,0.001472,1.666540e+05,1.661440e+05,1.650111e+05,1.476618e+05,82.617623,97.218643,98.149864,0.000000,0.000000,0.000000,1.061224,0.383610,0.225611,0.275639,25.95,25.9250,0.000963,25.95,25.915000,0.001349,26.375000,25.8750,0.019268,25.928732,25.926038,25.939678,0.006250,0.001300,-0.001818,0.000151,0.000116,0.002263,25.920139,0.001152,25.924524,0.000982,0.016474,0.002890
2016-01-04 08:48:00,25.9375,25.9375,25.9375,25.9375,800.0,1.0,103.750000,25.942675,25.932325,-0.000482,-0.000482,0.0000,0.0000,0.0000,0.000000,25.94000,-0.000096,25.934167,0.000129,25.927381,0.000390,25.929800,0.000297,26.143699,-0.007887,25.940859,25.939335,25.935963,25.933526,25.933588,25.985710,43.992282,56.847801,48.713022,39.222311,0.000000,0.000482,0.000386,0.000868,13.625304,65.554010,105.555556,0.005605,0.006148,-0.000543,0.004901,0.004173,0.000728,-0.003918,-0.010015,0.006097,0.777778,0.925926,0.835165,0.945055,0.007637,0.000294,43.992261,56.007686,61.093650,0.004924,0.000190,55.635866,42.071181,41.548609,0.006751,0.000260,33.195038,34.659154,31.991850,0.010103,0.000390,20.221922,33.746611,50.551650,25.911392,25.944608,0.001281,25.903088,25.952912,0.001922,25.798057,26.061543,0.010161,25.933119,25.918586,25.947652,0.001121,25.933119,25.913742,25.952496,0.001494,1.658540e+05,1.662140e+05,1.656254e+05,1.494860e+05,61.297444,90.537724,94.646014,0.000000,0.000000,0.000000,1.325967,0.835821,0.366348,0.429159,25.95,25.9250,0.000964,25.95,25.916389,0.001296,26.368333,25.8750,0.019020,25.929386,25.926375,25.932053,0.002500,0.001264,-0.000783,0.000323,0.000152,0.002263,25.920196,0.000668,25.928600,0.000343,0.016964,0.002410
2016-01-04 08:49:00,25.9275,25.9500,25.9125,25.9250,15312.0,16.0,103.703726,25.930175,25.919825,-0.000482,-0.000482,-0.0025,0.0225,

In [3]:
# --- DIAGNOSTICS & FLAGGING ---
# Identifies DRIFT, CONST, or HIGH_NA features for special handling.
df_inds_flag, diag = feats.flag_indicators(
    df = df_inds_unsc,
    train_prop = params.train_prop,
    pct_shift_thresh = 0.16,
    frac_outside_thresh = 0.06,
    min_train_samples = 50,
    na_rate_thresh = 0.4,
    const_tol = 1e-12,
)

# Display diagnostic summary
display(diag.sort_values("status", ascending=False).head(10))

Flagging indicators:   0%|          | 0/130 [00:00<?, ?feat/s]

,feature,status,reason,pct_shift_val,pct_shift_te,frac_val_out,frac_te_out,na_rate_train,na_rate_all
4,volume,OK,,0.009137,0.010172,0.042845,0.054326,0.0,0.0
5,trade_count,OK,,0.013069,0.010890,0.024688,0.016161,0.0,0.0
9,ret,OK,,0.000000,0.000000,0.011721,0.018990,0.0,0.0
10,log_ret,OK,,0.000000,0.000000,0.011721,0.018990,0.0,0.0
12,upper_shad,OK,,0.000000,0.000000,0.017721,0.041656,0.0,0.0
13,lower_shad,OK,,0.000000,0.000000,0.019401,0.043891,0.0,0.0
14,range_pct,OK,,0.055228,0.014177,0.006858,0.011353,0.0,0.0
16,sma_pct_5,OK,,0.003075,0.000214,0.011547,0.018468,0.0,0.0
18,sma_pct_9,OK,,0.000525,0.008644,0.010982,0.018608,0.0,0.0
20,sma_pct_21,OK,,0.002549,0.013939,0.010358,0.018993,0.0,0.0


In [4]:
# --- APPLY ROBUST Z (RZ) TO DRIFTS ---
# Preserve raw prices for plotting/strategy verification later
df_inds_flag[[f"{c}_raw" for c in params.strategy_cols_tick]] = df_inds_unsc[params.strategy_cols_tick]
df_inds_flag["close_raw"] = df_inds_unsc["close"]

# Apply the RZ transformation specifically to flagged columns
df_inds_rz = feats.apply_rz_to_drifts(
    df = df_inds_flag,
    diag = diag,
    rz_window = 60,
    eps = 1e-6
)

df_inds_rz

Applying RZ to DRIFTs:   0%|          | 0/62 [00:00<?, ?feat/s]

,volume,trade_count,ret,log_ret,upper_shad,lower_shad,range_pct,sma_pct_5,sma_pct_9,sma_pct_21,sma_pct_50,sma_pct_100,rsi_3,rsi_7,rsi_14,rsi_21,roc_3,roc_5,roc_10,roc_21,cci_7,cci_14,cci_20,stoch_k_5_3_3,stoch_d_5_3_3,stoch_k_14_3_3,stoch_d_14_3_3,atr_pct_3,plus_di_3,minus_di_3,adx_3,atr_pct_7,plus_di_7,minus_di_7,adx_7,atr_pct_14,plus_di_14,minus_di_14,adx_14,atr_pct_21,plus_di_21,minus_di_21,adx_21,bb_w_20_2p0,bb_w_20_3p0,bb_w_50_2p0,kc_w_20_20_1.5,kc_w_20_20_2.0,mfi_7,mfi_14,mfi_20,cmf_7,cmf_14,cmf_20,vol_spike_3,vol_spike_7,vol_spike_14,vol_spike_28,donch_w_10,donch_w_20,donch_w_55,ret_std_5,ret_std_21,ret_std_63,vwap_dist_session,psar_dist,dist_high_100,dist_low_100,atr_21_raw,adx_21_raw,rsi_21_raw,vwap_ohlc_close_session_raw,close_raw,open_RZ,high_RZ,low_RZ,close_RZ,vwap_RZ,ask_RZ,bid_RZ,body_RZ,sma_5_RZ,sma_9_RZ,sma_21_RZ,sma_50_RZ,sma_100_RZ,ema_3_RZ,ema_5_RZ,ema_9_RZ,ema_13_RZ,ema_21_RZ,ema_50_RZ,macd_line_3_10_4_RZ,macd_signal_3_10_4_RZ,macd_diff_3_10_4_RZ,macd_line_6_13_5_RZ,macd_signal_6_13_5_RZ,macd_diff_6_13_5_RZ,macd_line_12_26_9_RZ,macd_signal_12_26_9_RZ,macd_diff_12_26_9_RZ,atr_3_RZ,atr_7_RZ,atr_14_RZ,atr_21_RZ,bb_lband_20_2p0_RZ,bb_hband_20_2p0_RZ,bb_lband_20_3p0_RZ,bb_hband_20_3p0_RZ,bb_lband_50_2p0_RZ,bb_hband_50_2p0_RZ,kc_mid_20_20_1.5_RZ,kc_l_20_20_1.5_RZ,kc_h_20_20_1.5_RZ,kc_mid_20_20_2.0_RZ,kc_l_20_20_2.0_RZ,kc_h_20_20_2.0_RZ,obv_RZ,obv_roll_3_RZ,obv_roll_7_RZ,obv_roll_21_RZ,donch_h_10_RZ,donch_l_10_RZ,donch_h_20_RZ,donch_l_20_RZ,donch_h_55_RZ,donch_l_55_RZ,roll_vwap_10_RZ,roll_vwap_20_RZ,roll_vwap_50_RZ,slope_close_5_RZ,slope_close_20_RZ,slope_close_50_RZ,vwap_ohlc_close_session_RZ,psar_RZ
2016-01-04 08:47:00,520.0,2.0,0.000241,0.000241,0.0000,0.0000,0.000000,0.000482,0.000653,0.000914,0.000452,-0.007566,96.817308,90.125845,59.208009,43.243449,0.000723,0.000868,0.000868,0.001109,131.154684,224.437928,262.411348,1.000000,0.888889,1.000000,0.971429,0.000201,96.817213,3.182669,85.632759,0.000141,87.291639,9.110761,46.159542,0.000243,38.254340,24.700467,34.286782,0.000385,21.487845,29.599048,51.826216,0.001306,0.001959,0.013784,0.001104,0.001472,82.617623,97.218643,98.149864,0.000000,0.000000,0.000000,1.061224,0.383610,0.225611,0.275639,0.000963,0.001349,0.019268,0.000151,0.000116,0.002263,0.001152,0.000982,0.016474,0.002890,0.009983,51.826216,43.243449,25.920139,25.9500,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2016-01-04 08:48:00,800.0,1.0,-0.000482,-0.000482,0.0000,0.0000,0.000000,-0.000096,0.000129,0.000390,0.000297,-0.007887,43.992282,56.847801,48.713022,39.222311,0.000000,0.000482,0.000386,0.000868,13.625304,65.554010,105.555556,0.777778,0.925926,0.835165,0.945055,0.000294,43.992261,56.007686,61.093650,0.000190,55.635866,42.071181,41.548609,0.000260,33.195038,34.659154,31.991850,0.000390,20.221922,33.746611,50.551650,0.001281,0.001922,0.010161,0.001121,0.001494,61.297444,90.537724,94.646014,0.000000,0.000000,0.000000,1.325967,0.835821,0.366348,0.429159,0.000964,0.001296,0.019020,0.000323,0.000152,0.002263,0.000668,0.000343,0.016964,0.002410,0.010103,50.551650,39.222311,25.920196,25.9375,-2.000000,-2.000000,-2.000000,-2.000000,0.000000,-2.000000,-2.000000,0.000000,2.000000,2.000000,2.000000,-2.000000,-2.000000,-2.000000,-2.000000,2.000000,2.000000,2.000000,-2.000000,-2.000000,-2.000000,-2.000000,-2.000000,2.000000,-2.000000,2.000000,2.000000,-2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,-2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,-

In [5]:
# --- MIN-MAX SCALING & TIME FEATURES ---
# Scale all features (including RZ) to [0,1] based on training distribution
df_feat_scal, stats = feats.scale_features(
    df=df_inds_rz,
    train_prop=params.train_prop,
    p_lo=1.0,
    p_hi=99.0,
    include_rz=True
)
stats

MinMax Scaling:   0%|          | 0/130 [00:00<?, ?feat/s]

,min,max
feature,,
volume,257.000000,1.117148e+06
trade_count,1.000000,3.982000e+03
ret,-0.001771,1.767095e-03
log_ret,-0.001773,1.765535e-03
upper_shad,0.000000,1.300000e-01
...,...,...
slope_close_5_RZ,-8.776007,9.000000e+00
slope_close_20_RZ,-8.342307,8.807574e+00
slope_close_50_RZ,-8.137803,8.468831e+00


In [6]:
df_time = feats.add_session_centered_time_features(df_feat_scal)
df_time

,time_minute,time_hour,time_dow,time_month,time_day_of_year,time_week_of_year,time_in_sess,time_premark,time_afthour
2016-01-04 08:47:00,0.303472,0.270833,0.500000,0.500000,0.508219,0.500000,0.0,1.0,0.0
2016-01-04 08:48:00,0.304167,0.270833,0.500000,0.500000,0.508219,0.500000,0.0,1.0,0.0
2016-01-04 08:49:00,0.304861,0.270833,0.500000,0.500000,0.508219,0.500000,0.0,1.0,0.0
2016-01-04 08:50:00,0.305556,0.270833,0.500000,0.500000,0.508219,0.500000,0.0,1.0,0.0
2016-01-04 08:51:00,0.306250,0.270833,0.500000,0.500000,0.508219,0.500000,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...
2026-03-13 15:41:00,0.590972,0.562500,0.071429,0.666667,0.694521,0.692308,1.0,0.0,0.0
2026-03-13 15:42:00,0.591667,0.562500,0.071429,0.666667,0.694521,0.692308,1.0,0.0,0.0
2026-03-13 15:43:00,0.592361,0.562500,0.071429,0.666667,0.694521,0.692308,1.0,0.0,0.0
2026-03-13 15:44:00,0.593056,0.562500,0.071429,0.666667,0.694521,0.692308,1.0,0.0,0.0


In [7]:
df_feat_scal = pd.concat([df_feat_scal, df_time], axis=1)
df_feat_scal

,volume,trade_count,ret,log_ret,upper_shad,lower_shad,range_pct,sma_pct_5,sma_pct_9,sma_pct_21,sma_pct_50,sma_pct_100,rsi_3,rsi_7,rsi_14,rsi_21,roc_3,roc_5,roc_10,roc_21,cci_7,cci_14,cci_20,stoch_k_5_3_3,stoch_d_5_3_3,stoch_k_14_3_3,stoch_d_14_3_3,atr_pct_3,plus_di_3,minus_di_3,adx_3,atr_pct_7,plus_di_7,minus_di_7,adx_7,atr_pct_14,plus_di_14,minus_di_14,adx_14,atr_pct_21,plus_di_21,minus_di_21,adx_21,bb_w_20_2p0,bb_w_20_3p0,bb_w_50_2p0,kc_w_20_20_1.5,kc_w_20_20_2.0,mfi_7,mfi_14,mfi_20,cmf_7,cmf_14,cmf_20,vol_spike_3,vol_spike_7,vol_spike_14,vol_spike_28,donch_w_10,donch_w_20,donch_w_55,ret_std_5,ret_std_21,ret_std_63,vwap_dist_session,psar_dist,dist_high_100,dist_low_100,atr_21_raw,adx_21_raw,rsi_21_raw,vwap_ohlc_close_session_raw,close_raw,open_RZ,high_RZ,low_RZ,close_RZ,vwap_RZ,ask_RZ,bid_RZ,body_RZ,sma_5_RZ,sma_9_RZ,sma_21_RZ,sma_50_RZ,sma_100_RZ,ema_3_RZ,ema_5_RZ,ema_9_RZ,ema_13_RZ,ema_21_RZ,ema_50_RZ,macd_line_3_10_4_RZ,macd_signal_3_10_4_RZ,macd_diff_3_10_4_RZ,macd_line_6_13_5_RZ,macd_signal_6_13_5_RZ,macd_diff_6_13_5_RZ,macd_line_12_26_9_RZ,macd_signal_12_26_9_RZ,macd_diff_12_26_9_RZ,atr_3_RZ,atr_7_RZ,atr_14_RZ,atr_21_RZ,bb_lband_20_2p0_RZ,bb_hband_20_2p0_RZ,bb_lband_20_3p0_RZ,bb_hband_20_3p0_RZ,bb_lband_50_2p0_RZ,bb_hband_50_2p0_RZ,kc_mid_20_20_1.5_RZ,kc_l_20_20_1.5_RZ,kc_h_20_20_1.5_RZ,kc_mid_20_20_2.0_RZ,kc_l_20_20_2.0_RZ,kc_h_20_20_2.0_RZ,obv_RZ,obv_roll_3_RZ,obv_roll_7_RZ,obv_roll_21_RZ,donch_h_10_RZ,donch_l_10_RZ,donch_h_20_RZ,donch_l_20_RZ,donch_h_55_RZ,donch_l_55_RZ,roll_vwap_10_RZ,roll_vwap_20_RZ,roll_vwap_50_RZ,slope_close_5_RZ,slope_close_20_RZ,slope_close_50_RZ,vwap_ohlc_close_session_RZ,psar_RZ,time_minute,time_hour,time_dow,time_month,time_day_of_year,time_week_of_year,time_in_sess,time_premark,time_afthour
2016-01-04 08:47:00,0.000235,0.000251,0.568673,0.569106,0.000000,0.000000,0.000000,0.625118,0.617515,0.604468,0.537265,0.145141,0.968173,0.903736,0.592269,0.385027,0.617371,0.608483,0.577922,0.569920,0.827744,0.969336,1.000000,1.000000,0.888889,1.000000,0.971429,0.062041,0.968304,0.031838,0.821628,0.044863,0.914523,0.096586,0.380528,0.083018,0.457397,0.303614,0.321154,0.137632,0.248309,0.398007,0.674196,0.075003,0.075003,0.541332,0.131051,0.131051,0.826176,0.972186,0.981499,0.500000,0.500130,0.500653,0.404908,0.091412,0.043448,0.046360,0.077947,0.074984,0.717843,0.068029,0.053566,1.000000,0.547688,0.573531,0.696254,0.126473,0.009983,51.826216,43.243449,25.920139,25.9500,0.470169,0.448904,0.498787,0.471218,0.494730,0.471126,0.471200,0.500000,0.473684,0.477012,0.479256,0.475462,0.490903,0.472149,0.473562,0.474042,0.475296,0.478562,0.487237,0.488860,0.486769,0.497824,0.484518,0.482683,0.493139,0.486335,0.489646,0.487147,0.127322,0.150441,0.170387,0.181595,0.621238,0.343150,0.693088,0.271876,0.660979,0.296382,0.478818,0.526104,0.423193,0.478818,0.545772,0.409447,0.324983,0.327751,0.333903,0.350406,0.419916,0.541399,0.492341,0.467980,0.814815,0.071429,0.512665,0.515093,0.509213,0.493700,0.486435,0.490033,0.459549,0.487260,0.303472,0.270833,0.500000,0.500000,0.508219,0.500000,0.0,1.0,0.0
2016-01-04 08:48:00,0.000486,0.000000,0.364453,0.364863,0.000000,0.000000,0.000000,0.475912,0.523860,0.546369,0.526550,0.129635,0.439922,0.566988,0.470745,0.330657,0.499336,0.559746,0.534687,0.555082,0.534795,0.637187,0.707205,0.777778,0.925926,0.835165,0.945055,0.092263,0.439982,0.560283,0.516970,0.061801,0.581985,0.450114,0.326947,0.089338,0.392552,0.435853,0.291600,0.139458,0.229490,0.462517,0.654800,0.073428,0.073428,0.395104,0.133209,0.133209,0.612974,0.905377,0.946460,0.500000,0.500130,0.500653,0.514395,0.207041,0.071745,0.072578,0.077987,0.071710,0.708411,0.145706,0.070576,1.000000,0.532295,0.525210,0.716958,0.105445,0.010103,50.551650,39.222311,25.920196,25.9375,0.362702,0.342411,0.391586,0.362939,0.494730,0.362871,0.362916,0.500000,0.577624,0.578523,0.579670,0.351758,0.335298,0.367206,0.369981,0.575802,0.576547,0.581231,0.374751,0.364881,0.364365,0.383810,0.361090,0.603349,0.365042,0.605703,0.604874,0.343298,0.210770

In [12]:
importlib.reload(plots)
plots.plot_dual_histograms(
    df_before = df_inds_rz,
    df_after  = df_feat_scal,
)

In [9]:
# Final Save
params.to_parquet_with_progress(df_feat_scal, params.feat_all_parquet)
print(f"✅ Feature Engineering Complete. File saved to {params.feat_all_parquet}")

Saving Parquet: 100%|██████████| 94/94 [00:19<00:00,  4.85it/s]


✅ Feature Engineering Complete. File saved to dfs/AAPL_3_feat_all.parquet


In [13]:
# --- DAILY PLOTS CHECK ---
# Filter for your check month (defined in params.py)
df_month = df_feat_scal[df_feat_scal.index.to_period('M') == params.month_to_check].copy()

for base_day, grp in df_month.groupby(df_month.index.normalize()):
    print('─' * 100)
    print(f"Chart for: {base_day.date()} | Features: {len(grp.columns)}")
    plots.plot_trades(
        df=grp,
        col_close='close_raw',
        features=list(grp.columns),
    )